# Collect revision outputs for the repository

Copies only the *result* files (JSON, CSV, PNG, TXT) of the revision from Drive into one clean folder `results_revision/`, writes a manifest with file counts and sizes, and zips it. It never copies datasets (files named `data_*.csv`, `Dataset_*.csv`), model caches, backups or checkpoints. Read-only on Drive; run once after notebooks 01-07 (and the per-file cell of notebook 05) are done.

Output: `/content/results_revision/` (+ `results_revision.zip`, saved to Drive and offered for download).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob, csv, json, zipfile

DR = '/content/drive/MyDrive/Detector_Results'
JS = '/content/drive/MyDrive/JamShield_Dataset'
DEST = '/content/results_revision'
EXT = ('.json', '.csv', '.png', '.txt')
SKIP_NAME = ('data_', 'Dataset_')           # never copy dataset files
SKIP_DIR = ('cadangan', 'backup', '__pycache__', '.ipynb_checkpoints')

# (source folder, destination under results_revision)
MAP = [
    (f'{DR}/factorial',               'factorial'),                  # RF-Stat / LSTM / BERT cells A-D, 15 and 25 m/s
    (f'{DR}/pilot_bert_maxlen288',    'factorial_bert_maxlen288'),   # BERT C, D at MAX_LEN=288
    (f'{DR}/e41_hybrid',              'e41_hybrid'),                 # hybrid vs baselines (15 m/s)
    (f'{DR}/e41_hybrid_25',           'e41_hybrid_25ms'),            # only if it was run
    (f'{DR}/knn_sensitivity',         'knn_sensitivity'),
    (f'{DR}/knn_sensitivity_ext',     'knn_sensitivity_ext'),
    (f'{DR}/audit_e55',               'audit_e55'),
    (f'{DR}/tahap_lanjutan',          'tables_and_latency'),         # CSV tables, latency_suite*.json, corrected-statistics tables
    (f'{JS}/results',                 'jamshield/rf_lstm_main'),
    (f'{JS}/results_e59',             'jamshield/e59_counter_fix'),
    (f'{JS}/results_e510_v2',         'jamshield/e510_lofo_sgkf'),   # includes LOFO_per_file.csv/json
    (f'{JS}/results_e510_v3_manual',  'jamshield/e510_manual_folds'),
    (f'{JS}/results_e511_v2',         'jamshield/e511'),
]
EXTRA_FILES = [(f'{JS}/E5_8_inventaris_file.csv', 'jamshield/E5_8_file_inventory.csv')]

In [ ]:
if os.path.exists(DEST): shutil.rmtree(DEST)
manifest, missing = [], []
def keep(path):
    n = os.path.basename(path)
    return n.endswith(EXT) and not n.startswith(SKIP_NAME)
for src, dst in MAP:
    if not os.path.isdir(src):
        missing.append(src); continue
    for root, dirs, files in os.walk(src):
        dirs[:] = [d for d in dirs if not d.startswith(SKIP_DIR)]
        for f in files:
            p = os.path.join(root, f)
            if not keep(p): continue
            rel = os.path.relpath(p, src)
            q = os.path.join(DEST, dst, rel)
            os.makedirs(os.path.dirname(q), exist_ok=True)
            shutil.copy2(p, q)
            manifest.append((os.path.relpath(q, DEST), os.path.getsize(q)))
for src, dst in EXTRA_FILES:
    if os.path.exists(src):
        q = os.path.join(DEST, dst); os.makedirs(os.path.dirname(q), exist_ok=True); shutil.copy2(src, q)
        manifest.append((dst, os.path.getsize(q)))
    else:
        missing.append(src)

print('NOT FOUND (skipped):'); [print('  ', m) for m in missing]
print()
tot = {}
for rel, sz in manifest:
    top = '/'.join(rel.split('/')[:2]) if rel.startswith('jamshield') else rel.split('/')[0]
    c, s = tot.get(top, (0, 0)); tot[top] = (c + 1, s + sz)
print(f'{"folder":40s} {"files":>6s} {"MB":>8s}')
for k, (c, s) in sorted(tot.items()): print(f'{k:40s} {c:6d} {s/1e6:8.2f}')
total_mb = sum(s for _, s in manifest) / 1e6
print(f'{"TOTAL":40s} {len(manifest):6d} {total_mb:8.2f}')
big = [(r, s / 1e6) for r, s in manifest if s > 20e6]
print('\nFiles > 20 MB (check before uploading):', big if big else 'none')
with open(f'{DEST}/MANIFEST.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['path', 'bytes']); w.writerows(sorted(manifest))

In [ ]:
# Sanity: expected per-seed counts (20 seeds). Compare with the table above before uploading.
def n_json(sub): return len(glob.glob(f'{DEST}/{sub}/*.json'))
print('factorial JSON files        :', n_json('factorial'), '(RF 4 cells + LSTM 4 + BERT 4, x 20 seeds x 2 regimes, minus unrun cells)')
print('BERT @288 JSON files        :', n_json('factorial_bert_maxlen288'))
print('e41_hybrid seed JSON files  :', len(glob.glob(f'{DEST}/e41_hybrid/E41_15ms_seed*.json')), '(expected 20)')
print('JamShield E5.9 JSON files   :', n_json('jamshield/e59_counter_fix'), '(expected 20)')
print('LOFO_per_file present       :', os.path.exists(f'{DEST}/jamshield/e510_lofo_sgkf/LOFO_per_file.csv'))
print('latency JSON present        :', sorted(os.path.basename(p) for p in glob.glob(f'{DEST}/tables_and_latency/latency_suite*.json')))
leak = [r for r, _ in manifest if os.path.basename(r).startswith(SKIP_NAME)]
assert not leak, f'dataset files in output: {leak}'

In [ ]:
ZIP = '/content/results_revision.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(DEST):
        for f in files:
            p = os.path.join(root, f); z.write(p, os.path.relpath(p, '/content'))
print(f'zip: {os.path.getsize(ZIP)/1e6:.1f} MB')
shutil.copy(ZIP, '/content/drive/MyDrive/results_revision.zip')   # also kept on Drive
from google.colab import files as colab_files
colab_files.download(ZIP)